# ViT-only AI-vs-Real fine-tuning and source-separated evaluation (v5)

This Colab notebook fine-tunes **only the Hugging Face ViT** from the existing v3 checkpoint. It does not load, train, modify, or evaluate the CNN. It writes all outputs to a new v5 run directory, leaving v3 and original Drive files untouched.

## Dataset sources and terms checked

- **GenImage official benchmark** — [official repository](https://github.com/GenImage-Dataset/GenImage), [owner's shared Drive](https://drive.google.com/drive/folders/1jGt10bwTbhEZuGXLyvrCuxOI0cBqQ1FS?usp=sharing), [official project page](https://genimage-dataset.github.io/). The project page states CC BY-NC-SA 4.0 **plus additional Dataset Terms**, which prevail in case of conflict and restrict use to non-commercial academic research/teaching/publication. Attribute/cite the authors, comply with the extra terms, and do not redistribute the dataset. Official layout is generator / `train|val` / `ai|nature`; this notebook reserves ADM for validation and Midjourney for final testing.
- **Optional CIFAKE auxiliary data** — [Hugging Face dataset card](https://huggingface.co/datasets/yanbax/CIFAKE_autotrain_compatible), which describes 50,000 real CIFAR-10 images and 50,000 Stable Diffusion 1.4 images, and declares MIT for its published copy. The card says this mirror removes the original test split; therefore it is used only as optional training data, never validation/test. Confirm the original image rights and attribution before use. CIFAKE images are 32x32 and one-generator-specific, so they may add domain shortcuts and are not a substitute for diverse high-resolution real photographs. Optional and disabled by default; set `CIFAKE_AUX_PER_CLASS` above zero to upload it.
- **CIFAR-10 source** — [official dataset page](https://www.cs.toronto.edu/~kriz/cifar.html); 32x32 real images, and the page requests citation. The page itself does not provide a clear license for every underlying image; do not use or redistribute independently of the CIFAKE card without confirming applicable terms.

Dataset images come from ZIP file(s) selected from your local computer in the Colab upload dialog, not from Google Drive. The notebook inventories the uploaded archive contents and stops with a clear error if required generators/classes are absent. It does not inspect metadata for labels or model features. Image filenames and C2PA/EXIF/XMP are not classifier inputs.

**90% is a goal to investigate, not a promise.** The final Midjourney/source-separated score is evaluated only after model selection and must not be used to tune training. A good random-split score is not evidence of unseen-generator generalization.

### Colab setup and local ZIP upload

1. In Colab select **Runtime → Change runtime type → GPU** (T4 or better recommended).
2. Optional CIFAKE: set `CIFAKE_AUX_PER_CLASS` above zero in the configuration cell; the notebook then prompts for one local CIFAKE ZIP. Leave it at zero to skip CIFAKE.
3. Run all cells. The data cell opens a local file picker: select the GenImage ZIP archive(s) from your computer. For multiple archives, upload one ZIP per generator and keep generator names in their filenames (for example `BigGAN.zip`, `ADM.zip`, `Midjourney.zip`). A single ZIP may contain all generators, provided each image path includes its generator folder and `train|val/ai|nature` folders. Upload ordinary ZIPs containing the image files/folders; nested ZIP bundles and split-volume archives must be unpacked/repackaged first. Upload may be slow or fail for very large archives; Google Drive is no longer used as the image source.
4. Put `ai_vs_real_genimage_outputs_v3.zip` in My Drive (the notebook prompts for it if it cannot find it). Only the `huggingface_vit` model and processor are extracted. Drive is still mounted for the starting checkpoint and versioned results/resume state, not for dataset images.
5. Run all cells. Each epoch saves a resumable state to My Drive. After a disconnect/runtime reset, upload the dataset ZIP(s) again; the split signature selects the matching run state.

Exported package name starts `ai_vs_real_vit_generalization_v5_` and is saved under `My Drive/ai_vs_real_vit_genimage_outputs_v5/` and downloaded at the end. To deploy, extract `huggingface_vit/` into a separate app artifact directory and point `Config.HUGGINGFACE_FINETUNED_MODEL_DIR` there; keep v3 as a backup and confirm the app's `REAL`/`FAKE` label mapping before switching.


## 1. Install dependencies and configure this ViT-only run

In [ ]:
import shutil
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.46,<5", "safetensors", "scikit-learn", "pandas",
        "pillow", "ImageHash", "tqdm",
    ],
    check=True,
)
if shutil.which("7z") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "p7zip-full"], check=True)


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import random
import shutil
import subprocess
import time
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path, PurePosixPath

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import imagehash
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFile, UnidentifiedImageError
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tqdm.auto import tqdm
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    get_cosine_schedule_with_warmup,
)
from google.colab import drive, files

SEED = 42
TRAIN_PER_CLASS = 50_000
VALIDATION_PER_CLASS = 5_000
TEST_PER_CLASS = 5_000
TRAIN_GENERATORS = [
    "BigGAN", "VQDM", "wukong", "glide",
    "stable_diffusion_v_1_4", "stable_diffusion_v_1_5",
]
REAL_TRAIN_GENERATOR = "BigGAN"
VALIDATION_GENERATOR = "ADM"
TEST_GENERATOR = "Midjourney"
CLASS_FOLDERS = {0: "nature", 1: "ai"}

# Optional CIFAKE auxiliary training data. Leave None to use only the uploaded GenImage ZIP(s).
# When enabled, the notebook prompts for a local CIFAKE ZIP if this is None.
# Accepted structure: train/real and train/fake (or Real/Fake class folders).
# It is never used for validation or final testing.
CIFAKE_AUXILIARY_PATH = None
CIFAKE_AUX_PER_CLASS = 0
GENIMAGE_UPLOAD_ROOT = Path("/content/genimage_v5_uploaded_archives")

VIT_BATCH_SIZE = 16
VIT_EPOCHS = 5
VIT_LEARNING_RATE = 1e-5
VIT_WEIGHT_DECAY = 0.01
VIT_GRAD_ACCUMULATION_STEPS = 2
VIT_EARLY_STOPPING_PATIENCE = 2
VIT_MAX_GRAD_NORM = 1.0
PHASH_MAX_DISTANCE = 4
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
V3_ZIP_NAME = "ai_vs_real_genimage_outputs_v3.zip"
DOWNLOAD_ZIP_AT_END = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device={DEVICE}")
if DEVICE.type != "cuda":
    raise RuntimeError("A Colab GPU is required. Select Runtime > Change runtime type > GPU.")
GPU = torch.cuda.get_device_properties(0)
gpu_free_bytes, gpu_total_bytes = torch.cuda.mem_get_info(0)
page_size = os.sysconf("SC_PAGE_SIZE")
ram_total_bytes = page_size * os.sysconf("SC_PHYS_PAGES")
ram_available_bytes = page_size * os.sysconf("SC_AVPHYS_PAGES")
print(f"GPU: {GPU.name} | free={gpu_free_bytes / 2**30:.1f} GiB / total={gpu_total_bytes / 2**30:.1f} GiB")
print(f"Host RAM: available={ram_available_bytes / 2**30:.1f} GiB / total={ram_total_bytes / 2**30:.1f} GiB")
print(f"Free Colab disk: {shutil.disk_usage('/content').free / 2**30:.1f} GiB")
ImageFile.LOAD_TRUNCATED_IMAGES = False


## 2. Mount Drive and load the starting ViT checkpoint

GenImage image archives are uploaded from your computer in the next cell and unpacked selectively to Colab's local disk. Drive is used only for the v3 ViT checkpoint and versioned run/output files. The new run is stored in a split-signature-specific v5 folder for safe resume without replacing prior checkpoints.

In [ ]:
drive.mount("/content/drive")
MYDRIVE = Path("/content/drive/MyDrive")
if not MYDRIVE.is_dir():
    raise RuntimeError("Google Drive is not mounted. Rerun this cell and approve Drive access.")

REQUIRED_GENERATORS = list(dict.fromkeys(
    TRAIN_GENERATORS + [VALIDATION_GENERATOR, TEST_GENERATOR]
))
GENIMAGE_UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)

def local_archive_upload(prompt, upload_root, required=True):
    """Upload local ZIP archive(s) into Colab storage without changing source files."""
    upload_root.mkdir(parents=True, exist_ok=True)
    archives = sorted(upload_root.glob("*.zip"))
    if not archives:
        print(prompt)
        uploaded = files.upload()
        for filename, contents in uploaded.items():
            safe_name = Path(filename).name
            if Path(safe_name).suffix.lower() != ".zip":
                raise ValueError(f"Expected a .zip archive, received {safe_name!r}.")
            target = upload_root / safe_name
            if target.exists():
                raise FileExistsError(f"Refusing to overwrite an uploaded archive: {target}")
            with target.open("wb") as archive_file:
                archive_file.write(contents)
        del uploaded
        archives = sorted(upload_root.glob("*.zip"))
    if required and not archives:
        raise FileNotFoundError("No local ZIP was uploaded. Rerun this cell and choose dataset ZIP(s).")
    for archive in archives:
        if not zipfile.is_zipfile(archive):
            raise ValueError(f"Uploaded file is not a readable ZIP archive: {archive.name}")
        print(f"Uploaded archive: {archive.name} ({archive.stat().st_size / 2**30:.2f} GiB)")
    return archives


def normalized_name(value):
    return "".join(character for character in value.casefold() if character.isalnum())


GENIMAGE_ARCHIVES = local_archive_upload(
    "Choose the GenImage ZIP file(s) from your computer. For multiple archives, "
    "select the generator ZIPs together.",
    GENIMAGE_UPLOAD_ROOT,
)
if len(GENIMAGE_ARCHIVES) == 1:
    GENIMAGE_ARCHIVES_BY_GENERATOR = {
        generator: GENIMAGE_ARCHIVES for generator in REQUIRED_GENERATORS
    }
else:
    GENIMAGE_ARCHIVES_BY_GENERATOR = {}
    for generator in REQUIRED_GENERATORS:
        matches = [
            archive for archive in GENIMAGE_ARCHIVES
            if normalized_name(generator) in normalized_name(archive.stem)
        ]
        if len(matches) != 1:
            raise ValueError(
                f"Could not uniquely match an uploaded ZIP to generator {generator!r}; "
                f"matching files: {[archive.name for archive in matches]}. "
                "Upload one combined dataset ZIP or name each ZIP with its generator."
            )
        GENIMAGE_ARCHIVES_BY_GENERATOR[generator] = matches

UPLOAD_SOURCE_ROOT = Path("/content/genimage_v5_upload_sources")
UPLOAD_SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
GEN_DIRS = {}
for generator, archives in GENIMAGE_ARCHIVES_BY_GENERATOR.items():
    generator_dir = UPLOAD_SOURCE_ROOT / generator
    generator_dir.mkdir(parents=True, exist_ok=True)
    for archive in archives:
        link = generator_dir / archive.name
        if not link.exists():
            link.symlink_to(archive)
    GEN_DIRS[generator] = generator_dir
print("GenImage images will be read from local Colab uploads:")
for generator, archives in GENIMAGE_ARCHIVES_BY_GENERATOR.items():
    print(f"  {generator}: {[archive.name for archive in archives]}")

# The run folder is signature-specific; reruns never delete or overwrite another run.
OUTPUT_ROOT = Path("/content/ai_vs_real_vit_genimage_outputs_v5")
DRIVE_OUTPUT_ROOT = MYDRIVE / "ai_vs_real_vit_genimage_outputs_v5"
LOCAL_ROOT = Path("/content/genimage_v5_extracted")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)


def safe_extract_zip(archive_path, destination):
    """Extract a ZIP to Colab local storage only; reject traversal and symlinks."""
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    marker = destination / ".complete.json"
    archive_stat = archive_path.stat()
    signature = {"size": archive_stat.st_size, "mtime_ns": archive_stat.st_mtime_ns}
    if marker.is_file() and json.loads(marker.read_text()) == signature:
        print(f"Reusing local extraction: {destination}")
        return destination
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            parts = PurePosixPath(member.filename).parts
            if PurePosixPath(member.filename).is_absolute() or ".." in parts:
                raise ValueError(f"Unsafe path in ZIP member: {member.filename}")
            unix_mode = member.external_attr >> 16
            if unix_mode & 0o170000 == 0o120000:
                raise ValueError(f"Symlink in ZIP is not supported: {member.filename}")
            target = (destination / Path(*parts)).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f"ZIP member escapes extraction directory: {member.filename}")
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open("wb") as output:
                shutil.copyfileobj(source, output)
    marker.write_text(json.dumps(signature), encoding="utf-8")
    print(f"Extracted {archive_path.name} to {destination}; source archive was not changed.")
    return destination


CIFAKE_AUX_ROOT = None
if CIFAKE_AUXILIARY_PATH:
    cifake_path = Path(CIFAKE_AUXILIARY_PATH)
    if not cifake_path.exists():
        raise FileNotFoundError(f"Configured CIFAKE path does not exist: {cifake_path}")
    if cifake_path.is_file() and cifake_path.suffix.lower() == ".zip":
        archive_key = hashlib.sha256(str(cifake_path).encode()).hexdigest()[:12]
        CIFAKE_AUX_ROOT = safe_extract_zip(cifake_path, Path("/content/cifake_aux") / archive_key)
    elif cifake_path.is_dir():
        CIFAKE_AUX_ROOT = cifake_path
    else:
        raise ValueError("CIFAKE_AUXILIARY_PATH must be a directory or a standard .zip archive.")
    print(f"CIFAKE auxiliary source: {CIFAKE_AUX_ROOT} (training split only)")
elif CIFAKE_AUX_PER_CLASS > 0:
    cifake_upload_root = Path("/content/cifake_v5_uploaded_archives")
    cifake_upload_root.mkdir(parents=True, exist_ok=True)
    cifake_archives = local_archive_upload(
        "Optional: choose a CIFAKE ZIP from your computer, or cancel/leave empty to skip CIFAKE.",
        cifake_upload_root,
        required=False,
    )
    if cifake_archives:
        if len(cifake_archives) != 1:
            raise ValueError("Upload exactly one CIFAKE ZIP archive.")
        archive_digest = hashlib.sha256()
        with cifake_archives[0].open("rb") as archive_file:
            for chunk in iter(lambda: archive_file.read(1 << 20), b""):
                archive_digest.update(chunk)
        archive_key = archive_digest.hexdigest()[:12]
        CIFAKE_AUX_ROOT = safe_extract_zip(
            cifake_archives[0], Path("/content/cifake_aux") / archive_key
        )
        print(f"CIFAKE auxiliary source: {CIFAKE_AUX_ROOT} (training split only)")
else:
    print("No CIFAKE auxiliary data configured; using GenImage sources only.")

CIFAKE_AUX_PER_CLASS = CIFAKE_AUX_PER_CLASS if CIFAKE_AUX_ROOT else 0
GENIMAGE_TRAIN_PER_CLASS = TRAIN_PER_CLASS - CIFAKE_AUX_PER_CLASS
if GENIMAGE_TRAIN_PER_CLASS <= 0:
    raise ValueError("Auxiliary CIFAKE quota must be smaller than TRAIN_PER_CLASS.")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def find_v3_zip():
    folders = [
        MYDRIVE / "AI_vs_Real", MYDRIVE / "ai_vs_real_genimage_outputs_v3",
        MYDRIVE, MYDRIVE / "Colab Notebooks",
    ]
    for folder in folders:
        if folder.is_dir():
            matches = sorted(folder.glob("ai_vs_real_genimage_outputs_v3*.zip"))
            if matches:
                return matches[0]
    return None


PREV_DIR = Path("/content/previous_v3_outputs")
VIT_CHECKPOINT_DIR = PREV_DIR / "huggingface_vit"

def v3_vit_ready():
    required = ("config.json", "model.safetensors", "preprocessor_config.json")
    return all((VIT_CHECKPOINT_DIR / filename).is_file() for filename in required)


def extract_v3_vit(zip_path):
    PREV_DIR.mkdir(parents=True, exist_ok=True)
    root = PREV_DIR.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            if member.is_dir():
                continue
            parts = PurePosixPath(member.filename).parts
            if PurePosixPath(member.filename).is_absolute() or ".." in parts:
                raise ValueError(f"Unsafe path in starting checkpoint ZIP: {member.filename}")
            index = next((i for i, part in enumerate(parts) if part == "huggingface_vit"), None)
            if index is None:
                continue
            target = (PREV_DIR / Path(*parts[index:])).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f"Unsafe model path in ZIP: {member.filename}")
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open("wb") as output:
                shutil.copyfileobj(source, output)


if not v3_vit_ready():
    v3_zip = find_v3_zip()
    if v3_zip is None:
        print(f"Upload {V3_ZIP_NAME}; only the ViT checkpoint and processor will be used.")
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(zip_names) != 1:
            raise ValueError(f"Upload exactly one v3 checkpoint ZIP; received {list(uploaded)}")
        v3_zip = Path("/content") / zip_names[0]
        input_archive_dir = MYDRIVE / "ai_vs_real_vit_inputs"
        input_archive_dir.mkdir(parents=True, exist_ok=True)
        content_hash = hashlib.sha256(v3_zip.read_bytes()).hexdigest()[:12]
        preserved_copy = input_archive_dir / f"starting_v3_{content_hash}.zip"
        if not preserved_copy.exists():
            shutil.copy2(v3_zip, preserved_copy)
        v3_zip = preserved_copy
    print(f"Starting checkpoint archive: {v3_zip}")
    extract_v3_vit(v3_zip)
if not v3_vit_ready():
    raise FileNotFoundError(
        "The starting ZIP must contain huggingface_vit/config.json, model.safetensors, "
        "and preprocessor_config.json."
    )
checkpoint_digest = hashlib.sha256()
for checkpoint_file in sorted(path for path in VIT_CHECKPOINT_DIR.rglob("*") if path.is_file()):
    checkpoint_digest.update(str(checkpoint_file.relative_to(VIT_CHECKPOINT_DIR)).encode("utf-8"))
    with checkpoint_file.open("rb") as source:
        for chunk in iter(lambda: source.read(1 << 20), b""):
            checkpoint_digest.update(chunk)
VIT_CHECKPOINT_SIGNATURE = checkpoint_digest.hexdigest()
print(f"ViT starting checkpoint: {VIT_CHECKPOINT_DIR}")
print(f"Starting checkpoint SHA-256: {VIT_CHECKPOINT_SIGNATURE}")


## 3. Inventory and extract the requested GenImage subsets

The archive-aware helper below extracts only the classes/splits selected for training, validation, and testing from standard ZIP files uploaded from your computer. The archive must contain actual image files, organized with generator, split, and class folders; a nested bundle of ZIPs is not unpacked automatically. It must see all six training sources, ADM, and Midjourney. CIFAKE, if enabled, is training-only.

In [ ]:
SEVENZ = shutil.which("7z")
if SEVENZ is None:
    raise RuntimeError("7z is unavailable. Re-run the dependency-install cell.")
SEVENZ = shutil.which("7z")

def find_master_zip(gdir):
    zips = sorted(p for p in gdir.iterdir() if p.is_file() and p.suffix.lower() == ".zip")
    for z in zips:                                     # prefer the one that has .z01 siblings
        if z.with_suffix(".z01").exists():
            return z
    return zips[0] if zips else None

def index_archive(archive, wanted, generator):
    """Index wanted split/classes only under this generator in an uploaded ZIP."""
    err = Path("/content/7z_list.err")
    found, prefixes = defaultdict(list), {}
    generator_key = normalized_name(generator)
    configured_generator_keys = {normalized_name(name) for name in REQUIRED_GENERATORS}
    archive_name_matches = generator_key in normalized_name(archive.stem)
    with err.open("w") as ef:
        proc = subprocess.Popen([SEVENZ, "l", "-slt", "-ba", str(archive)], stdout=subprocess.PIPE,
                                stderr=ef, text=True, errors="replace")
        cur, is_dir = None, False
        def flush():
            if not cur or is_dir or Path(cur).suffix.lower() not in IMAGE_EXTENSIONS:
                return
            parts = cur.split("/")
            normalized_parts = [normalized_name(part) for part in parts]
            generator_indexes = [
                index for index, part in enumerate(normalized_parts)
                if part in configured_generator_keys
            ]
            if generator_indexes:
                matching_indexes = [
                    index for index in generator_indexes if normalized_parts[index] == generator_key
                ]
                if not matching_indexes:
                    return
                candidate_indexes = [matching_indexes[0] + 1]
            elif archive_name_matches:
                candidate_indexes = range(len(parts) - 2)
            else:
                return
            for i in candidate_indexes:
                if i + 1 >= len(parts):
                    continue
                key = (parts[i].casefold(), parts[i + 1].casefold())
                wanted_key = next(
                    (pair for pair in wanted if tuple(item.casefold() for item in pair) == key),
                    None,
                )
                if wanted_key is not None:
                    prefix = "/".join(parts[:i])
                    if wanted_key not in prefixes:
                        prefixes[wanted_key] = prefix
                    if prefixes[wanted_key] == prefix:
                        found[wanted_key].append(cur)
                    break
        for line in proc.stdout:
            line = line.rstrip("\n")
            if line.startswith("Path = "):
                flush(); cur, is_dir = line[7:].replace("\\", "/"), False
            elif line.startswith("Folder = "):
                is_dir = line.endswith("+")
        flush()
        proc.wait()
    if proc.returncode not in (0, 1) and not found:
        raise RuntimeError(f"7z could not list {archive}:\n{err.read_text()[-1500:]}")
    return {key: (prefixes[key], values) for key, values in found.items()}

def build_patterns(prefix, split, label, names, need):
    """7z include patterns. If need is None (or the folder is small) take everything; otherwise take an
    evenly spread subset by last character of the file name (>= need*1.15 files)."""
    base = f"{prefix}/{split}/{label}" if prefix else f"{split}/{label}"
    if need is None or len(names) <= need * 1.3:
        return [f"{base}/*"], len(names)
    groups = defaultdict(int)
    for n in names:
        p = Path(n); groups[(p.stem[-1:], p.suffix)] += 1
    keys = sorted(groups); random.Random(SEED).shuffle(keys)
    picked, total = [], 0
    for k in keys:
        picked.append(k); total += groups[k]
        if total >= need * 1.15:
            break
    return [f"{base}/*{c}{e}" for c, e in picked], total

def run_7z(archive, out_dir, patterns, class_dir, expected, tag):
    log = Path(f"/content/7z_{tag}.log")
    for attempt in range(1, 4):
        with log.open("w") as lf:
            proc = subprocess.Popen([SEVENZ, "x", "-y", "-bso0", "-bsp0", str(archive), f"-o{out_dir}", *patterns],
                                    stdout=lf, stderr=subprocess.STDOUT)
            t0 = time.time(); last = 0.0
            while proc.poll() is None:
                time.sleep(2)
                if time.time() - last > 30:
                    last = time.time()
                    n = sum(1 for _ in class_dir.iterdir()) if class_dir.is_dir() else 0
                    print(f"   ... {n:,}/{expected:,} files, {(time.time()-t0)/60:.1f} min", flush=True)
        if proc.returncode in (0, 1):
            return
        print(f"   7z failed (exit {proc.returncode}), attempt {attempt}/3: {log.read_text()[-400:]}")
        time.sleep(10)
    raise RuntimeError(f"7z extraction failed for {tag}. See {log}")

ARCHIVE_INDEX, CLASS_DIRS = {}, {}

def prepare_generator(gen, jobs):
    """jobs: list of (split, label, need_or_None). Fills CLASS_DIRS[(gen,split,label)]."""
    gdir = GEN_DIRS[gen]
    pending = []
    for split, label, need in jobs:
        marker = LOCAL_ROOT / gen / f".done_{split}_{label}"
        if marker.is_file() and Path(marker.read_text().strip()).is_dir():
            CLASS_DIRS[(gen, split, label)] = Path(marker.read_text().strip())
            print(f"[Skipped] {gen}/{split}/{label} already extracted")
        else:
            pending.append((split, label, need))
    if not pending:
        return
    archive = find_master_zip(gdir)
    if archive is None:
        # Fallback for an already-extracted local dataset folder.
        for split, label, _ in pending:
            hits = [p for p in list(gdir.glob(f"{split}/{label}")) + list(gdir.glob(f"*/{split}/{label}")) if p.is_dir()]
            if not hits:
                raise FileNotFoundError(
                    f"No usable uploaded ZIP or extracted '{split}/{label}' folder for {gen} in {gdir}. "
                    "Check the ZIP contents and preserve the generator/train|val/ai|nature folders."
                )
            CLASS_DIRS[(gen, split, label)] = hits[0]
            print(f"[Local folder] {gen}/{split}/{label} -> {hits[0]}")
        return
    print(f"\n=== {gen}: listing {archive.name} ===", flush=True)
    t0 = time.time()
    index = index_archive(archive, {(s, l) for s, l, _ in pending}, gen)
    print(f"    listed in {time.time()-t0:.0f}s", flush=True)
    for split, label, need in pending:
        if (split, label) not in index:
            raise FileNotFoundError(
                f"{archive.name} has no {split}/{label} images for {gen}. The ZIP must contain "
                f"image files under a {gen}/{split}/{label} path (or be named for {gen} if "
                "it contains one generator at its root). Nested ZIP bundles are not supported; "
                "extract/repackage them on your computer first."
            )
        prefix, names = index[(split, label)]
        patterns, expected = build_patterns(prefix, split, label, names, need)
        out_dir = LOCAL_ROOT / gen
        class_dir = out_dir / prefix / split / label if prefix else out_dir / split / label
        print(f"-> {gen} {split}/{label}: {len(names):,} in archive, extracting about {expected:,} "
              f"(free disk {shutil.disk_usage('/content').free/1e9:.0f} GB)", flush=True)
        run_7z(archive, out_dir, patterns, class_dir, expected, f"{gen}_{split}_{label}")
        got = sum(1 for _ in class_dir.iterdir()) if class_dir.is_dir() else 0
        if got < max(1, int(expected * 0.9)):
            raise RuntimeError(f"Only {got:,} of ~{expected:,} files extracted for {gen}/{split}/{label}. "
                               f"Check /content/7z_{gen}_{split}_{label}.log (disk space or invalid archive?)")
        (LOCAL_ROOT / gen / f".done_{split}_{label}").write_text(str(class_dir))
        CLASS_DIRS[(gen, split, label)] = class_dir
        print(f"   done: {got:,} files in {class_dir}", flush=True)

# validation + test: take everything (about 6,000 per class); train: spread subset
prepare_generator(VALIDATION_GENERATOR, [("val", "nature", None), ("val", "ai", None)])
prepare_generator(TEST_GENERATOR,       [("val", "nature", None), ("val", "ai", None)])

TRAIN_QUOTAS = {
    generator: GENIMAGE_TRAIN_PER_CLASS // len(TRAIN_GENERATORS)
    + int(index < GENIMAGE_TRAIN_PER_CLASS % len(TRAIN_GENERATORS))
    for index, generator in enumerate(TRAIN_GENERATORS)
}
for generator, quota in TRAIN_QUOTAS.items():
    jobs = []
    if generator == REAL_TRAIN_GENERATOR:
        jobs.append(("train", "nature", math.ceil(GENIMAGE_TRAIN_PER_CLASS * 1.25)))
    jobs.append(("train", "ai", math.ceil(quota * 1.25)))
    prepare_generator(generator, jobs)

print("\nExtraction complete.")
for k, v in CLASS_DIRS.items():
    print(f"  {k}: {v}")

## 4. Validate data, filter duplicates, and save source-separated manifests

The order is held-out Midjourney test, ADM validation, then training. SHA-256 catches byte-identical copies; an image-only perceptual hash filters likely re-encodes/near duplicates. Labels come only from the documented class directories. Validation/test generator groups never enter training. All selected paths, labels, source groups, hashes, counts, exclusions, and seed are saved to Drive.

In [ ]:
class PerceptualHashIndex:
    """BK-tree index for finding perceptually similar images efficiently."""
    def __init__(self):
        self.root = None

    @staticmethod
    def distance(left, right):
        return (left ^ right).bit_count()

    def add(self, value, record):
        if self.root is None:
            self.root = [value, [record], {}]
            return
        node = self.root
        while True:
            distance = self.distance(value, node[0])
            if distance == 0:
                node[1].append(record)
                return
            if distance not in node[2]:
                node[2][distance] = [value, [record], {}]
                return
            node = node[2][distance]

    def find(self, value, radius):
        matches = []
        if self.root is None:
            return matches
        pending = [self.root]
        while pending:
            node = pending.pop()
            distance = self.distance(value, node[0])
            if distance <= radius:
                matches.extend(node[1])
            low, high = distance - radius, distance + radius
            pending.extend(child for edge, child in node[2].items() if low <= edge <= high)
        return matches


perceptual_index = PerceptualHashIndex()
seen_exact_hashes = {}
duplicate_counts = defaultdict(int)
unreadable_counts = defaultdict(int)


def inspect_file(path):
    """Return exact and pixel-derived perceptual hashes; metadata is not read."""
    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            pixel_hash = int(imagehash.phash(image.convert("RGB"), hash_size=8))
        digest = hashlib.sha256()
        with open(path, "rb") as image_file:
            for chunk in iter(lambda: image_file.read(1 << 20), b""):
                digest.update(chunk)
        return digest.hexdigest(), pixel_hash
    except (OSError, UnidentifiedImageError, Image.DecompressionBombError):
        return None


def list_images(class_dir):
    return sorted(
        str(path) for path in Path(class_dir).rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def select_unique_paths(paths, generator, source_split, label, count, split_name):
    paths = list(paths)
    random.Random(SEED + sum(map(ord, f"{generator}:{source_split}:{label}:{split_name}"))).shuffle(paths)
    selected = []
    with ThreadPoolExecutor(max_workers=8) as pool:
        for image_path, result in zip(paths, pool.map(inspect_file, paths)):
            if result is None:
                unreadable_counts[split_name] += 1
                continue
            digest, pixel_hash = result
            prior = seen_exact_hashes.get(digest)
            if prior is not None:
                if prior[1] != int(label):
                    raise ValueError(f"Byte-identical image has conflicting labels: {image_path}")
                duplicate_counts[f"exact_{split_name}"] += 1
                continue
            similar = perceptual_index.find(pixel_hash, PHASH_MAX_DISTANCE)
            if similar:
                if any(record[1] != int(label) for record in similar):
                    raise ValueError(f"Near-duplicate image has conflicting labels: {image_path}")
                duplicate_counts[f"perceptual_{split_name}"] += 1
                continue
            record = {
                "path": image_path,
                "label": int(label),
                "generator": generator,
                "source_split": source_split,
                "sha256": digest,
                "phash64": f"{pixel_hash:016x}",
            }
            seen_exact_hashes[digest] = (split_name, int(label))
            perceptual_index.add(pixel_hash, (split_name, int(label), image_path))
            selected.append(record)
            if len(selected) >= count:
                break
    if len(selected) < count:
        raise RuntimeError(
            f"Insufficient readable, nonduplicate images for {split_name}/{generator}/"
            f"{label}: requested {count:,}, found {len(selected):,}; "
            f"unreadable={unreadable_counts[split_name]}, duplicates={dict(duplicate_counts)}. "
            "Add eligible images or lower the requested count; the notebook will not silently "
            "train on a smaller split."
        )
    return pd.DataFrame(selected)


def select_unique(generator, source_split, label, count, split_name):
    class_dir = CLASS_DIRS[(generator, source_split, CLASS_FOLDERS[label])]
    paths = list_images(class_dir)
    print(f"Inventory {generator}/{source_split}/{CLASS_FOLDERS[label]}: {len(paths):,} files")
    return select_unique_paths(paths, generator, source_split, label, count, split_name)


def balanced(frame, split_name):
    counts = frame["label"].value_counts()
    if set(counts.index) != {0, 1}:
        raise RuntimeError(f"Split {split_name} is missing a class: {counts.to_dict()}")
    per_class = int(counts.min())
    parts = [
        frame[frame.label == label].sample(n=per_class, random_state=SEED)
        for label in (0, 1)
    ]
    result = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=SEED)
    return result.reset_index(drop=True)


def cifake_training_class_dirs(root):
    """Locate explicitly named Real/Fake folders; prefer train folders only."""
    class_names = {
        0: {"real", "real_images", "realimage", "nature"},
        1: {"fake", "fake_images", "ai", "ai_generated", "generated"},
    }
    directories = [path for path in root.rglob("*") if path.is_dir()]
    has_train_tree = any("train" in {part.casefold() for part in p.parts} for p in directories)
    output = {}
    for label, accepted_names in class_names.items():
        candidates = [
            directory for directory in directories
            if directory.name.casefold().replace(" ", "_") in accepted_names
            and (not has_train_tree or "train" in {part.casefold() for part in directory.parts})
            and any(file.suffix.lower() in IMAGE_EXTENSIONS for file in directory.rglob("*"))
        ]
        if not candidates:
            raise FileNotFoundError(
                f"CIFAKE auxiliary data has no explicit class folder for label {label}. "
                "Expected train/real and train/fake, or Real and Fake folders."
            )
        candidates.sort(key=lambda directory: len(directory.parts), reverse=True)
        selected = []
        for candidate in candidates:
            if not any(candidate == existing or candidate in existing.parents for existing in selected):
                selected.append(candidate)
        output[label] = [file for folder in selected for file in folder.rglob("*")
                         if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS]
    return output


test_parts = [
    select_unique(TEST_GENERATOR, "val", label, TEST_PER_CLASS, "test")
    for label in (0, 1)
]
test_frame = balanced(pd.concat(test_parts, ignore_index=True), "test")
validation_parts = [
    select_unique(VALIDATION_GENERATOR, "val", label, VALIDATION_PER_CLASS, "validation")
    for label in (0, 1)
]
validation_frame = balanced(pd.concat(validation_parts, ignore_index=True), "validation")

train_real = select_unique(
    REAL_TRAIN_GENERATOR, "train", 0, GENIMAGE_TRAIN_PER_CLASS, "train"
)
TRAIN_QUOTAS = {
    generator: GENIMAGE_TRAIN_PER_CLASS // len(TRAIN_GENERATORS)
    + int(index < GENIMAGE_TRAIN_PER_CLASS % len(TRAIN_GENERATORS))
    for index, generator in enumerate(TRAIN_GENERATORS)
}
train_ai_parts = [
    select_unique(generator, "train", 1, quota, "train")
    for generator, quota in TRAIN_QUOTAS.items()
]
train_parts = [train_real, *train_ai_parts]
if CIFAKE_AUX_ROOT is not None:
    cifake_dirs = cifake_training_class_dirs(CIFAKE_AUX_ROOT)
    for label in (0, 1):
        print(f"CIFAKE label {label} explicit folder inventory: {len(cifake_dirs[label]):,}")
        aux_frame = select_unique_paths(
            cifake_dirs[label], "CIFAKE_auxiliary", "train", label,
            CIFAKE_AUX_PER_CLASS, "train",
        )
        train_parts.append(aux_frame)
train_frame = balanced(pd.concat(train_parts, ignore_index=True), "train")

for name, frame in (("train", train_frame), ("validation", validation_frame), ("test", test_frame)):
    counts = frame.label.value_counts().sort_index().to_dict()
    sources = frame.generator.value_counts().to_dict()
    print(f"{name:<10} {len(frame):>7,} images | classes={counts} | sources={sources}")
    assert counts.get(0) == counts.get(1), f"Unbalanced {name} split: {counts}"

assert set(validation_frame.generator) == {VALIDATION_GENERATOR}
assert set(test_frame.generator) == {TEST_GENERATOR}
assert not ({VALIDATION_GENERATOR, TEST_GENERATOR} & set(train_frame.generator))
assert set(train_frame.label) == set(validation_frame.label) == set(test_frame.label) == {0, 1}

signature_rows = pd.concat([
    frame.assign(dataset_split=split)[["dataset_split", "generator", "label", "sha256", "phash64"]]
    for split, frame in (("train", train_frame), ("validation", validation_frame), ("test", test_frame))
], ignore_index=True).sort_values(["dataset_split", "generator", "label", "sha256"])
SPLIT_SIGNATURE = hashlib.sha256(signature_rows.to_csv(index=False).encode()).hexdigest()
print(f"Selected-data signature: {SPLIT_SIGNATURE}")
print(f"Filtered duplicate candidates: {dict(duplicate_counts)}")
print(f"Unreadable images encountered: {dict(unreadable_counts)}")

TRAINING_CONFIG = {
    "model": "Hugging Face ViT",
    "starting_checkpoint": V3_ZIP_NAME,
    "starting_checkpoint_sha256": VIT_CHECKPOINT_SIGNATURE,
    "seed": SEED,
    "train_per_class": TRAIN_PER_CLASS,
    "validation_per_class": VALIDATION_PER_CLASS,
    "test_per_class": TEST_PER_CLASS,
    "epochs": VIT_EPOCHS,
    "batch_size": VIT_BATCH_SIZE,
    "learning_rate": VIT_LEARNING_RATE,
    "weight_decay": VIT_WEIGHT_DECAY,
    "gradient_accumulation_steps": VIT_GRAD_ACCUMULATION_STEPS,
    "max_grad_norm": VIT_MAX_GRAD_NORM,
    "early_stopping_patience": VIT_EARLY_STOPPING_PATIENCE,
    "mixed_precision": True,
    "generators": TRAIN_GENERATORS,
    "cifake_aux_per_class": CIFAKE_AUX_PER_CLASS,
}
TRAINING_CONFIG_SIGNATURE = hashlib.sha256(
    json.dumps(TRAINING_CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()
RUN_SIGNATURE = f"{SPLIT_SIGNATURE[:16]}-{TRAINING_CONFIG_SIGNATURE[:8]}"
RUN_DIR = OUTPUT_ROOT / RUN_SIGNATURE
DRIVE_RUN_DIR = DRIVE_OUTPUT_ROOT / RUN_SIGNATURE
if DRIVE_RUN_DIR.is_dir() and not RUN_DIR.exists():
    shutil.copytree(DRIVE_RUN_DIR, RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)
VIT_OUTPUT_DIR = RUN_DIR / "huggingface_vit"
METRICS_DIR = RUN_DIR / "metrics"
SPLITS_DIR = RUN_DIR / "splits"
RESUME_DIR = RUN_DIR / "resume"
for folder in (VIT_OUTPUT_DIR, METRICS_DIR, SPLITS_DIR, RESUME_DIR):
    folder.mkdir(parents=True, exist_ok=True)
for split_name, split_frame in (
    ("train", train_frame), ("validation", validation_frame), ("test", test_frame)
):
    split_frame.to_csv(SPLITS_DIR / f"{split_name}.csv", index=False)


def sync_to_drive():
    shutil.copytree(RUN_DIR, DRIVE_RUN_DIR, dirs_exist_ok=True)
    print(f"Saved resumable run to {DRIVE_RUN_DIR}")


source_manifest = {
    "official_genimage_repository": "https://github.com/GenImage-Dataset/GenImage",
    "genimage_source": "local computer upload to Colab /content",
    "genimage_upload_directory": str(GENIMAGE_UPLOAD_ROOT),
    "genimage_archives_by_generator": {
        name: [path.name for path in archives]
        for name, archives in GENIMAGE_ARCHIVES_BY_GENERATOR.items()
    },
    "cifake_source_path": str(CIFAKE_AUX_ROOT) if CIFAKE_AUX_ROOT else None,
    "genimage_license": "CC BY-NC-SA 4.0 plus additional dataset terms; non-commercial research only",
    "cifake_source": (
        "https://huggingface.co/datasets/yanbax/CIFAKE_autotrain_compatible"
        if CIFAKE_AUX_ROOT else None
    ),
    "cifake_license_note": (
        "Dataset card declares MIT and states real=CIFAR-10, fake=Stable Diffusion 1.4; "
        "verify underlying image rights and cite Bird & Lotfi and CIFAR-10. Training only."
        if CIFAKE_AUX_ROOT else None
    ),
    "label_mapping": {"nature/real": 0, "real": 0, "ai/fake": 1},
    "seed": SEED,
    "requested_images_per_class": {
        "train": TRAIN_PER_CLASS, "validation": VALIDATION_PER_CLASS, "test": TEST_PER_CLASS,
    },
    "actual_counts": {
        name: {str(label): int(count) for label, count in frame.label.value_counts().items()}
        for name, frame in (("train", train_frame), ("validation", validation_frame), ("test", test_frame))
    },
    "sources_by_split": {
        name: {str(source): int(count) for source, count in frame.generator.value_counts().items()}
        for name, frame in (("train", train_frame), ("validation", validation_frame), ("test", test_frame))
    },
    "split_signature_sha256": SPLIT_SIGNATURE,
    "training_config_signature": TRAINING_CONFIG_SIGNATURE,
    "training_config": TRAINING_CONFIG,
    "run_signature": RUN_SIGNATURE,
    "exact_and_perceptual_duplicate_counts": dict(duplicate_counts),
    "unreadable_counts": dict(unreadable_counts),
    "near_duplicate_method": f"ImageHash pHash; Hamming distance <= {PHASH_MAX_DISTANCE}",
    "pixel_input_only": True,
}
(METRICS_DIR / "dataset_manifest.json").write_text(
    json.dumps(source_manifest, indent=2) + "\n", encoding="utf-8"
)
sync_to_drive()


## 5. Fine-tune and resume only the Hugging Face ViT

The checkpoint's REAL/FAKE class mapping is verified before training. Validation balanced accuracy selects the checkpoint and validation scores select the threshold. Model, optimizer, scheduler, scaler, epoch, and history are checkpointed each epoch to Drive; resume is refused if the split signature differs. The final Midjourney test is evaluated only after selection.

In [ ]:
cached_selection_path = METRICS_DIR / "vit_stage.json"
latest_state_path = RESUME_DIR / "trainer_state.pt"
test_scores_path = METRICS_DIR / "vit_test_scores.npy"
test_labels_path = METRICS_DIR / "vit_test_labels.npy"
completed = (
    cached_selection_path.is_file()
    and test_scores_path.is_file()
    and test_labels_path.is_file()
    and (VIT_OUTPUT_DIR / "model.safetensors").is_file()
    and json.loads(cached_selection_path.read_text()).get("split_signature") == SPLIT_SIGNATURE
    and json.loads(cached_selection_path.read_text()).get("training_config_signature")
    == TRAINING_CONFIG_SIGNATURE
)

if completed:
    vit_selection = json.loads(cached_selection_path.read_text(encoding="utf-8"))
    vit_history = json.loads((METRICS_DIR / "vit_training_history.json").read_text())
    vit_test_scores = np.load(test_scores_path)
    vit_test_true = np.load(test_labels_path)
    print("This exact split already has a completed ViT run; loading its recorded outputs.")
else:
    processor = AutoImageProcessor.from_pretrained(str(VIT_CHECKPOINT_DIR), local_files_only=True)
    vit_model = AutoModelForImageClassification.from_pretrained(
        str(VIT_CHECKPOINT_DIR), local_files_only=True, use_safetensors=True
    )
    id2label = {
        int(index): str(label).strip().upper()
        for index, label in vit_model.config.id2label.items()
    }
    real_ids = [index for index, label in id2label.items() if label == "REAL"]
    fake_ids = [index for index, label in id2label.items() if label == "FAKE"]
    if len(real_ids) != 1 or len(fake_ids) != 1 or vit_model.config.num_labels != 2:
        raise ValueError(
            f"Starting checkpoint must have exactly REAL and FAKE classes; got {id2label}. "
            "Do not guess a class-index mapping."
        )
    REAL_ID, FAKE_ID = real_ids[0], fake_ids[0]
    label_to_model_id = {0: REAL_ID, 1: FAKE_ID}
    print(f"Verified checkpoint class mapping: {id2label}; dataset->model={label_to_model_id}")

    class ImageFrameDataset(torch.utils.data.Dataset):
        def __init__(self, frame):
            self.paths = frame.path.tolist()
            self.labels = frame.label.to_numpy(dtype=np.int64)

        def __len__(self):
            return len(self.paths)

        def __getitem__(self, index):
            image_path = self.paths[index]
            try:
                with Image.open(image_path) as image:
                    pixel_values = processor(
                        images=image.convert("RGB"), return_tensors="pt"
                    )["pixel_values"].squeeze(0)
            except (OSError, UnidentifiedImageError) as error:
                raise RuntimeError(f"Cannot decode selected training image {image_path}: {error}") from error
            return pixel_values, label_to_model_id[int(self.labels[index])]

    def collate_images(batch):
        images, labels = zip(*batch)
        return torch.stack(images), torch.tensor(labels, dtype=torch.long)

    worker_count = min(4, max(1, (os.cpu_count() or 2) // 2))
    use_amp = DEVICE.type == "cuda"

    def make_loader(frame, shuffle):
        return torch.utils.data.DataLoader(
            ImageFrameDataset(frame), batch_size=VIT_BATCH_SIZE, shuffle=shuffle,
            num_workers=worker_count, pin_memory=True, collate_fn=collate_images,
            persistent_workers=worker_count > 0, prefetch_factor=2 if worker_count else None,
        )

    train_loader = make_loader(train_frame, True)
    validation_loader = make_loader(validation_frame, False)
    test_loader = make_loader(test_frame, False)
    vit_model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        vit_model.parameters(), lr=VIT_LEARNING_RATE, weight_decay=VIT_WEIGHT_DECAY
    )
    updates_per_epoch = math.ceil(len(train_loader) / VIT_GRAD_ACCUMULATION_STEPS)
    total_steps = updates_per_epoch * VIT_EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=max(1, int(total_steps * 0.05)),
        num_training_steps=total_steps,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    def score_vit(model, loader, description):
        model.eval()
        labels, scores = [], []
        loss_total = 0.0
        with torch.inference_mode():
            for pixels, targets in tqdm(loader, desc=description, leave=False):
                pixels = pixels.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    output = model(pixel_values=pixels, labels=targets)
                probabilities = torch.softmax(output.logits.float(), dim=-1)[:, FAKE_ID]
                labels.extend((targets == FAKE_ID).long().cpu().tolist())
                scores.extend(probabilities.cpu().tolist())
                loss_total += float(output.loss.item()) * len(targets)
        return (
            np.asarray(labels, dtype=np.int64),
            np.asarray(scores, dtype=np.float64),
            loss_total / len(labels),
        )

    def metric_at_threshold(labels, scores, threshold=0.5):
        predicted = (scores >= threshold).astype(np.int64)
        return float(balanced_accuracy_score(labels, predicted)), float(accuracy_score(labels, predicted))

    history_path = METRICS_DIR / "vit_training_history.json"
    vit_history = []
    start_epoch = 0
    best_balanced_accuracy = -1.0
    best_validation_loss = float("inf")
    stale_epochs = 0
    if latest_state_path.is_file():
        state = torch.load(latest_state_path, map_location="cpu", weights_only=False)
        if state.get("split_signature") != SPLIT_SIGNATURE:
            raise RuntimeError("Saved resume state belongs to another data split; refusing to reuse it.")
        if state.get("training_config_signature") != TRAINING_CONFIG_SIGNATURE:
            raise RuntimeError(
                "Training settings differ from the saved resume state. Use a new run "
                "configuration instead of resuming incompatible optimizer state."
            )
        vit_model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        scheduler.load_state_dict(state["scheduler_state_dict"])
        scaler.load_state_dict(state["scaler_state_dict"])
        start_epoch = int(state["completed_epoch"])
        best_balanced_accuracy = float(state["best_balanced_accuracy"])
        best_validation_loss = float(state["best_validation_loss"])
        vit_history = state["history"]
        stale_epochs = int(state.get("stale_epochs", 0))
        random.setstate(state["python_rng_state"])
        np.random.set_state(state["numpy_rng_state"])
        torch.set_rng_state(state["torch_rng_state"])
        torch.cuda.set_rng_state_all(state["cuda_rng_state"])
        print(f"Resuming after epoch {start_epoch}; optimizer/scheduler/scaler/RNG restored.")
    else:
        starting_labels, starting_scores, starting_loss = score_vit(
            vit_model, validation_loader, "Starting checkpoint validation"
        )
        best_balanced_accuracy, _ = metric_at_threshold(starting_labels, starting_scores)
        best_validation_loss = starting_loss
        vit_model.config.id2label = {REAL_ID: "REAL", FAKE_ID: "FAKE"}
        vit_model.config.label2id = {"REAL": REAL_ID, "FAKE": FAKE_ID}
        vit_model.save_pretrained(VIT_OUTPUT_DIR, safe_serialization=True)
        processor.save_pretrained(VIT_OUTPUT_DIR)
        print(
            f"Starting checkpoint validation: balanced_accuracy={best_balanced_accuracy:.3%}, "
            f"loss={starting_loss:.5f}"
        )

    for epoch in range(start_epoch, VIT_EPOCHS):
        vit_model.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_loss = 0.0
        seen = 0
        for batch_index, (pixels, targets) in enumerate(
            tqdm(train_loader, desc=f"ViT epoch {epoch + 1}/{VIT_EPOCHS}")
        ):
            pixels = pixels.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                output = vit_model(pixel_values=pixels, labels=targets)
                loss = output.loss / VIT_GRAD_ACCUMULATION_STEPS
            scaler.scale(loss).backward()
            if (
                (batch_index + 1) % VIT_GRAD_ACCUMULATION_STEPS == 0
                or batch_index + 1 == len(train_loader)
            ):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(vit_model.parameters(), VIT_MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
            epoch_loss += float(output.loss.item()) * len(targets)
            seen += len(targets)
        val_labels, val_scores, val_loss = score_vit(
            vit_model, validation_loader, f"Validation epoch {epoch + 1}"
        )
        val_balanced_accuracy, val_accuracy = metric_at_threshold(val_labels, val_scores)
        record = {
            "epoch": epoch + 1,
            "train_loss": epoch_loss / seen,
            "validation_loss": val_loss,
            "validation_accuracy_at_0_5": val_accuracy,
            "validation_balanced_accuracy_at_0_5": val_balanced_accuracy,
            "learning_rate": scheduler.get_last_lr()[0],
        }
        vit_history.append(record)
        print(record)
        improved = (
            val_balanced_accuracy > best_balanced_accuracy
            or (val_balanced_accuracy == best_balanced_accuracy and val_loss < best_validation_loss)
        )
        if improved:
            best_balanced_accuracy = val_balanced_accuracy
            best_validation_loss = val_loss
            stale_epochs = 0
            vit_model.config.id2label = {REAL_ID: "REAL", FAKE_ID: "FAKE"}
            vit_model.config.label2id = {"REAL": REAL_ID, "FAKE": FAKE_ID}
            vit_model.save_pretrained(VIT_OUTPUT_DIR, safe_serialization=True)
            processor.save_pretrained(VIT_OUTPUT_DIR)
        else:
            stale_epochs += 1

        history_path.write_text(json.dumps(vit_history, indent=2) + "\n", encoding="utf-8")
        temporary_state_path = latest_state_path.with_name(latest_state_path.name + ".tmp")
        torch.save(
            {
                "split_signature": SPLIT_SIGNATURE,
                "training_config_signature": TRAINING_CONFIG_SIGNATURE,
                "completed_epoch": epoch + 1,
                "stale_epochs": stale_epochs,
                "python_rng_state": random.getstate(),
                "numpy_rng_state": np.random.get_state(),
                "torch_rng_state": torch.get_rng_state(),
                "cuda_rng_state": torch.cuda.get_rng_state_all(),
                "best_balanced_accuracy": best_balanced_accuracy,
                "best_validation_loss": best_validation_loss,
                "model_state_dict": vit_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "history": vit_history,
            },
            temporary_state_path,
        )
        os.replace(temporary_state_path, latest_state_path)
        sync_to_drive()
        if stale_epochs >= VIT_EARLY_STOPPING_PATIENCE:
            print("Early stopping: validation balanced accuracy did not improve.")
            break

    # Select threshold from validation only. The test set is evaluated once below.
    selected_model = AutoModelForImageClassification.from_pretrained(
        str(VIT_OUTPUT_DIR), local_files_only=True, use_safetensors=True
    ).to(DEVICE)
    val_labels, val_scores, _ = score_vit(
        selected_model, validation_loader, "Final validation threshold selection"
    )
    from sklearn.metrics import roc_curve

    false_positive_rate, true_positive_rate, candidate_thresholds = roc_curve(
        val_labels, val_scores
    )
    validation_balanced_accuracy = (
        true_positive_rate + 1.0 - false_positive_rate
    ) / 2.0
    finite_indices = np.flatnonzero(np.isfinite(candidate_thresholds))
    threshold_candidates = [
        (float(candidate_thresholds[index]), float(validation_balanced_accuracy[index]))
        for index in finite_indices
    ]
    threshold_candidates.append((0.5, metric_at_threshold(val_labels, val_scores, 0.5)[0]))
    best_validation_balanced_accuracy = max(score for _, score in threshold_candidates)
    tied_thresholds = [
        threshold for threshold, score in threshold_candidates
        if np.isclose(score, best_validation_balanced_accuracy)
    ]
    selected_threshold = min(tied_thresholds, key=lambda value: abs(value - 0.5))
    vit_test_true, vit_test_scores, test_loss = score_vit(
        selected_model, test_loader, f"Final held-out {TEST_GENERATOR} test"
    )
    np.save(test_scores_path, vit_test_scores)
    np.save(test_labels_path, vit_test_true)
    vit_selection = {
        "split_signature": SPLIT_SIGNATURE,
        "training_config_signature": TRAINING_CONFIG_SIGNATURE,
        "validation_generator": VALIDATION_GENERATOR,
        "starting_checkpoint": str(VIT_CHECKPOINT_DIR),
        "best_validation_balanced_accuracy_at_0_5": best_balanced_accuracy,
        "best_validation_loss": best_validation_loss,
        "selected_threshold_from_validation": selected_threshold,
        "validation_threshold_selection_metric": "maximum balanced accuracy; ties nearest 0.5",
        "completed_epochs": len(vit_history),
        "resume_enabled": True,
    }
    cached_selection_path.write_text(json.dumps(vit_selection, indent=2) + "\n")
    del selected_model, vit_model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()
    sync_to_drive()


## 6. Evaluate once on the held-out generator

The selected threshold comes from ADM validation. Report accuracy, balanced accuracy, precision, recall, F1, ROC-AUC, confusion matrix, and per-class metrics overall and per test source. Results apply to this held-out source; they do not establish 90% accuracy on other generators.

In [ ]:
def binary_metrics(labels, scores, threshold):
    labels = np.asarray(labels, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if len(labels) != len(scores) or len(labels) == 0 or not np.isfinite(scores).all():
        raise ValueError("Invalid labels/scores in ViT evaluation.")
    predictions = (scores >= threshold).astype(np.int64)
    return {
        "n": int(len(labels)),
        "real_count": int(np.sum(labels == 0)),
        "ai_generated_count": int(np.sum(labels == 1)),
        "threshold_from_validation": float(threshold),
        "accuracy": float(accuracy_score(labels, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "precision_ai": float(precision_score(labels, predictions, zero_division=0)),
        "recall_ai": float(recall_score(labels, predictions, zero_division=0)),
        "f1_ai": float(f1_score(labels, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(labels, scores)) if len(np.unique(labels)) == 2 else None,
        "confusion_matrix_real_ai": confusion_matrix(labels, predictions, labels=[0, 1]).tolist(),
        "per_class": classification_report(
            labels, predictions, labels=[0, 1], target_names=["real", "ai_generated"],
            output_dict=True, zero_division=0,
        ),
    }


expected_labels = test_frame.label.to_numpy(dtype=np.int64)
if not np.array_equal(vit_test_true, expected_labels):
    raise RuntimeError("Saved test outputs do not match the current held-out test manifest.")
threshold = float(vit_selection["selected_threshold_from_validation"])
overall_test_metrics = binary_metrics(vit_test_true, vit_test_scores, threshold)
per_source_metrics = {}
for source_name, source_frame in test_frame.groupby("generator", sort=True):
    row_indices = source_frame.index.to_numpy()
    per_source_metrics[str(source_name)] = binary_metrics(
        vit_test_true[row_indices], vit_test_scores[row_indices], threshold
    )

results = {
    "model": "Fine-tuned Hugging Face ViT only",
    "test_generator": TEST_GENERATOR,
    "test_metrics": overall_test_metrics,
    "per_generator_metrics": per_source_metrics,
    "selected_threshold": threshold,
    "threshold_selection_source": VALIDATION_GENERATOR,
    "interpretation": (
        "This is performance on the named held-out source only; it is not evidence of "
        "90% generalization to arbitrary images or unseen generators. Test results were "
        "not used for checkpoint, threshold, or hyperparameter selection."
    ),
}
(METRICS_DIR / "vit_test_metrics.json").write_text(
    json.dumps(results, indent=2) + "\n", encoding="utf-8"
)
summary_rows = [{"source": "ALL_TEST", **{
    key: value for key, value in overall_test_metrics.items()
    if key not in {"confusion_matrix_real_ai", "per_class"}
}}]
for source_name, metrics in per_source_metrics.items():
    summary_rows.append({"source": source_name, **{
        key: value for key, value in metrics.items()
        if key not in {"confusion_matrix_real_ai", "per_class"}
    }})
results_table = pd.DataFrame(summary_rows)
display(results_table.style.format({
    "accuracy": "{:.2%}", "balanced_accuracy": "{:.2%}",
    "precision_ai": "{:.2%}", "recall_ai": "{:.2%}", "f1_ai": "{:.2%}",
    "roc_auc": "{:.4f}",
}))
print("Held-out test confusion matrix [[real->real, real->AI], [AI->real, AI->AI]]:")
print(overall_test_metrics["confusion_matrix_real_ai"])
print("Per-class metrics:")
print(json.dumps(overall_test_metrics["per_class"], indent=2))
print("Threshold was selected using validation only:", threshold)
sync_to_drive()


## 7. Export only the selected ViT, processor, manifests, and metrics

The ZIP excludes optimizer resume state and contains no CNN files. The app expects the Hugging Face `config.json`, model weights, and matching processor in one directory; preserve the current v3 directory before switching.

In [ ]:
# Create a clean deliverable ZIP: selected ViT + matching processor + manifests/metrics only.
EXPORT_ROOT = Path(f"/content/vit_generalization_export_{RUN_SIGNATURE}")
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(VIT_OUTPUT_DIR, EXPORT_ROOT / "huggingface_vit", dirs_exist_ok=True)
shutil.copytree(SPLITS_DIR, EXPORT_ROOT / "splits", dirs_exist_ok=True)
shutil.copytree(METRICS_DIR, EXPORT_ROOT / "metrics", dirs_exist_ok=True)
run_metadata = {
    **source_manifest,
    "model": "Fine-tuned Hugging Face ViT",
    "starting_checkpoint_archive": V3_ZIP_NAME,
    "starting_checkpoint_sha256": VIT_CHECKPOINT_SIGNATURE,
    "training_generators": TRAIN_GENERATORS,
    "real_training_source": REAL_TRAIN_GENERATOR,
    "validation_generator": VALIDATION_GENERATOR,
    "held_out_test_generator": TEST_GENERATOR,
    "training_hyperparameters": {
        "epochs_max": VIT_EPOCHS,
        "batch_size": VIT_BATCH_SIZE,
        "gradient_accumulation_steps": VIT_GRAD_ACCUMULATION_STEPS,
        "learning_rate": VIT_LEARNING_RATE,
        "weight_decay": VIT_WEIGHT_DECAY,
        "early_stopping_patience": VIT_EARLY_STOPPING_PATIENCE,
        "device": torch.cuda.get_device_name(0),
    },
    "checkpoint_selection": vit_selection,
    "training_config": TRAINING_CONFIG,
    "test_results": results,
}
(EXPORT_ROOT / "run_metadata.json").write_text(
    json.dumps(run_metadata, indent=2) + "\n", encoding="utf-8"
)
zip_stem = f"ai_vs_real_vit_generalization_v5_{RUN_SIGNATURE}"
zip_path = Path(shutil.make_archive(f"/content/{zip_stem}", "zip", EXPORT_ROOT))
output_zip = DRIVE_OUTPUT_ROOT / zip_path.name
revision = 2
while output_zip.exists():
    output_zip = DRIVE_OUTPUT_ROOT / f"{zip_stem}_rerun{revision}.zip"
    revision += 1
shutil.copy2(zip_path, output_zip)
print(f"Exported ViT-only package: {output_zip}")
print(f"Size: {zip_path.stat().st_size / 2**20:.1f} MiB")
if DOWNLOAD_ZIP_AT_END:
    files.download(str(zip_path))
